# Install and import libraries

In [1]:
!pip -q install sacremoses

from pathlib import Path
from sacremoses import MosesTokenizer
import zipfile
import math


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. Gather plain text data in multiple languages, save each in separate files, one file per language.

I create a data folder under which I then download the public zips for each language I have chosen. The zips are stored in a public repository of my github. The languages I chose are English, Spanish, Czech and Polish. The zips contain UTF-8 encoded text files of different books which I have downloaded from https://www.gutenberg.org.

In [3]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

!wget -O data/english.zip "https://github.com/Cenkyyy/Language-Identification/releases/download/v1.0/english.zip"
!wget -O data/spanish.zip "https://github.com/Cenkyyy/Language-Identification/releases/download/v1.0/spanish.zip"
!wget -O data/czech.zip "https://github.com/Cenkyyy/Language-Identification/releases/download/v1.0/czech.zip"
!wget -O data/polish.zip "https://github.com/Cenkyyy/Language-Identification/releases/download/v1.0/polish.zip"

--2026-03-29 21:26:17--  https://github.com/Cenkyyy/Language-Identification/releases/download/v1.0/english.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1195159878/259c8d34-a15e-488f-986b-2ff5f96ac73b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-29T22%3A20%3A17Z&rscd=attachment%3B+filename%3Denglish.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-29T21%3A19%3A48Z&ske=2026-03-29T22%3A20%3A17Z&sks=b&skv=2018-11-09&sig=fZZfzCETNFElzac16fI26qCkUrEpAi5qo450cHM8g%2FM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDgxOTg3NywibmJmIjoxNzc0ODE5NTc3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvb

# 2. Tokenize everything and report the size of the data in tokens and bytes

I have created two helper methods, a `merge_files` method that takes a string to a zip file, extracts it and merges all the files inside it into a string which is then returned and a `tokenize` which tokenizes the texts into tokens and returns.

I then print out the sizes of tokens and bytes for respective languages.

In [4]:
def merge_files(zip_path):
    # string to store the text of all files
    merged_text = ""

    # extract the zip and get the files in them
    with zipfile.ZipFile(zip_path, "r") as zip:
        files = zip.namelist()

        # iterate through each file, read its text and add it to the merged text
        for file in files:
            text = zip.read(file).decode("utf-8")
            merged_text += text + "\n"

    return merged_text

def tokenize(text, language_code):
    tokenizer = MosesTokenizer(lang=language_code)
    tokens = tokenizer.tokenize(text, return_str=False, escape=False)
    return " ".join(tokens)

# create the texts for respective languages
english_corpus = merge_files("data/english.zip")
spanish_corpus = merge_files("data/spanish.zip")
czech_corpus = merge_files("data/czech.zip")
polish_corpus = merge_files("data/polish.zip")

# tokenize the texts for respective languages
english_corpus_tokenized = tokenize(english_corpus, "en")
spanish_corpus_tokenized = tokenize(spanish_corpus, "es")
czech_corpus_tokenized = tokenize(czech_corpus, "cs")
polish_corpus_tokenized = tokenize(polish_corpus, "pl")

# print out the tokens and bytes for respective languages
print(f"English corpus: {len(english_corpus_tokenized.split())} tokens, {len(english_corpus_tokenized.encode('utf-8'))} bytes")
print(f"Spanish corpus: {len(spanish_corpus_tokenized.split())} tokens, {len(spanish_corpus_tokenized.encode('utf-8'))} bytes")
print(f"Czech corpus: {len(czech_corpus_tokenized.split())} tokens, {len(czech_corpus_tokenized.encode('utf-8'))} bytes")
print(f"Polish corpus: {len(polish_corpus_tokenized.split())} tokens, {len(polish_corpus_tokenized.encode('utf-8'))} bytes")

English corpus: 972165 tokens, 4685808 bytes
Spanish corpus: 850543 tokens, 4189719 bytes
Czech corpus: 410391 tokens, 2273298 bytes
Polish corpus: 327996 tokens, 1955005 bytes


# 3. Split data into training, heldout and test sets. (80% of the data for training and 10% for heldout and test.)




In [5]:
def split_corpus(tokenized_text):
    tokens = tokenized_text.split()

    total = len(tokens)
    train_end = int(total * 0.8)
    heldout_end = int(total * 0.9)

    train_text = " ".join(tokens[:train_end])
    heldout_text = " ".join(tokens[train_end:heldout_end])
    test_text = " ".join(tokens[heldout_end:])

    return train_text, heldout_text, test_text

english_train, english_heldout, english_test = split_corpus(english_corpus_tokenized)
spanish_train, spanish_heldout, spanish_test = split_corpus(spanish_corpus_tokenized)
czech_train, czech_heldout, czech_test = split_corpus(czech_corpus_tokenized)
polish_train, polish_heldout, polish_test = split_corpus(polish_corpus_tokenized)

# 4. Estimate the unigram, bigram, and trigram probabilities of character n-grams in each language separately. (Use the conditional distributions for higher-level n-grams to be able to apply the Markovian property to estimate probabilities of sequences.)

In [6]:
# helper function to update the counts dictionary based on the key
def add_count(counts, key):
    if key in counts:
        counts[key] += 1
    else:
        counts[key] = 1

def estimate_character_ngram_probabilities(text):
    # store how many times each n-gram appears
    unigram_counts = {}
    bigram_counts = {}
    trigram_counts = {}

    # store the n-1 n-grams to compute the conditional probability correctly
    bigram_history_counts = {}
    trigram_history_counts = {}

    total_characters = len(text)

    # count the unigrams
    for character in text:
        add_count(unigram_counts, character)

    # count the bigrams and their histories
    for i in range(total_characters - 1):
        bigram = text[i:i + 2]
        bigram_history = text[i]

        add_count(bigram_counts, bigram)
        add_count(bigram_history_counts, bigram_history)

    # count the trigrams and their histories
    for i in range(total_characters - 2):
        trigram = text[i:i + 3]
        trigram_history = text[i:i + 2]

        add_count(trigram_counts, trigram)
        add_count(trigram_history_counts, trigram_history)

    # compute the unigram probabilities for each unigram as the count
    # of the unigram divided by total characters in the text
    unigram_probabilities = {}
    for unigram in unigram_counts:
        unigram_probabilities[unigram] = unigram_counts[unigram] / total_characters

    # compute the bigram probabilities for each bigram as the the conditional probability,
    # that means we divide the count of the bigram with the starting unigram from the bigram history
    bigram_probabilities = {}
    for bigram in bigram_counts:
        bigram_history = bigram[0]
        bigram_probabilities[bigram] = bigram_counts[bigram] / bigram_history_counts[bigram_history]

    # compute the trigram probabilities for each trigram as the conditional probability,
    # that means we divide the count of the trigram with the starting bigram from the trigram history
    trigram_probabilities = {}
    for trigram in trigram_counts:
        trigram_history = trigram[:2]
        trigram_probabilities[trigram] = trigram_counts[trigram] / trigram_history_counts[trigram_history]

    return {
        "total_characters": total_characters,
        "unigram_counts": unigram_counts,
        "bigram_counts": bigram_counts,
        "trigram_counts": trigram_counts,
        "bigram_history_counts": bigram_history_counts,
        "trigram_history_counts": trigram_history_counts,
        "unigram_probabilities": unigram_probabilities,
        "bigram_probabilities": bigram_probabilities,
        "trigram_probabilities": trigram_probabilities
    }

# pass the training data for each language and get their model, which in my case
# are the counts for n-grams, their probabilities and total characters
english_model = estimate_character_ngram_probabilities(english_train)
spanish_model = estimate_character_ngram_probabilities(spanish_train)
czech_model = estimate_character_ngram_probabilities(czech_train)
polish_model = estimate_character_ngram_probabilities(polish_train)

# 5. Report 5 most common character trigrams per language, along with their counts and relative frequencies (count divided by the size of the data).

In [7]:
def print_top_5_trigrams(language_name, model):
    # convert the model's trigram counts items to a list and sort them based on their count
    trigrams = list(model["trigram_counts"].items())
    trigrams.sort(key=lambda trigram: trigram[1], reverse=True)
    top_5_trigrams = trigrams[:5]

    print(f"{language_name} - 5 most common character trigrams:")

    # culculate the relative frequency of the trigram as its count divided by total characters for that language
    for trigram, count in top_5_trigrams:
        relative_frequency = count / model["total_characters"]
        print(f"{trigram}: count = {count}, relative frequency = {relative_frequency}")

    print()

print_top_5_trigrams("English", english_model)
print_top_5_trigrams("Spanish", spanish_model)
print_top_5_trigrams("Czech", czech_model)
print_top_5_trigrams("Polish", polish_model)

English - 5 most common character trigrams:
 th: count = 72299, relative frequency = 0.019389306180420125
 , : count = 58600, relative frequency = 0.015715477975803527
the: count = 56054, relative frequency = 0.015032686048731926
he : count = 47917, relative frequency = 0.012850487340726581
nd : count = 27148, relative frequency = 0.007280610854728911

Spanish - 5 most common character trigrams:
 , : count = 53706, relative frequency = 0.016088335163164526
 de: count = 40547, relative frequency = 0.012146384498209362
que: count = 32964, relative frequency = 0.009874797607689185
 qu: count = 32142, relative frequency = 0.009628556749980154
ue : count = 30193, relative frequency = 0.009044708292954725

Czech - 5 most common character trigrams:
 , : count = 29583, relative frequency = 0.01771806340482927
 . : count = 15193, relative frequency = 0.00909950097385563
 se: count = 11355, relative frequency = 0.0068008183741282615
se : count = 10379, relative frequency = 0.006216265425370077
 

# 6. Estimate the "add less than one" smoothing parameter for the trigram language model.

In [8]:
# computes the smoothed probability of the trigram (this is used by the heldout/test data, so the texts are new)
def smoothed_trigram_probability(trigram, model, _lambda):
    # take the history of the given trigram (which are the first two characters)
    trigram_history = trigram[:2]

    # get the trigram's count and its history count from the training data
    trigram_count = model["trigram_counts"].get(trigram, 0)
    trigram_history_count = model["trigram_history_counts"].get(trigram_history, 0)

    # number unique characters inside the training data represents the vocabulary size
    vocabulary_size = len(model["unigram_counts"])

    # then return the exact formula which is mentioned on the slide 16 in lecture 2 to get the trigrams smoothed probability
    return (trigram_count + _lambda) / (trigram_history_count + _lambda * vocabulary_size)

# evaluates the cross entropy on a given text (heldout/test data) given the smoothing parameter _lambda on the trigram language model
def trigram_cross_entropy(text, model, _lambda):
    log_probability_sum_accumulator = 0.0
    trigram_count = 0

    # iterate through each trigram in the given text
    for i in range(len(text) - 2):
        # get the trigram and increase the count
        trigram_count += 1
        trigram = text[i:i + 3]

        # calculate trigram's smoothed probability and accumulate it as the log of it
        probability = smoothed_trigram_probability(trigram, model, _lambda)
        log_probability_sum_accumulator += math.log2(probability)

    # return the average negarive log probability, which is the defition of the cross entorpy
    return -log_probability_sum_accumulator / trigram_count

# 7. Report the values of the smoothing parameters (one per language).

In [9]:
def estimate_best_lambda(model, heldout_text):
    # start with some initial value and calculate its cross entropy as a measurement
    best_lambda = 0.01
    best_cross_entropy = trigram_cross_entropy(heldout_text, model, best_lambda)

    # iterate with other candidates for the lambda value, calculate its cross entropy
    # and store for the best lambda and cross entropy if the current lambda has smaller
    # cross entropy than the best one, because we are trying to minimize the cross entropy
    for i in range(2, 100):
        _lambda = i / 100
        cross_entropy = trigram_cross_entropy(heldout_text, model, _lambda)

        if cross_entropy < best_cross_entropy:
            best_lambda = _lambda
            best_cross_entropy = cross_entropy

    return best_lambda, best_cross_entropy

# call the methods to calculate the best smoothing parameter and print it out for respective languages
english_lambda, english_heldout_cross_entropy = estimate_best_lambda(english_model, english_heldout)
spanish_lambda, spanish_heldout_cross_entropy = estimate_best_lambda(spanish_model, spanish_heldout)
czech_lambda, czech_heldout_cross_entropy = estimate_best_lambda(czech_model, czech_heldout)
polish_lambda, polish_heldout_cross_entropy = estimate_best_lambda(polish_model, polish_heldout)

print(f"English smoothing parameter: {english_lambda}")
print(f"Spanish smoothing parameter: {spanish_lambda}")
print(f"Czech smoothing parameter: {czech_lambda}")
print(f"Polish smoothing parameter: {polish_lambda}")

English smoothing parameter: 0.01
Spanish smoothing parameter: 0.04
Czech smoothing parameter: 0.04
Polish smoothing parameter: 0.02


# 8. Calculate the cross-entropies of all (trigram) language models on all test sets.

In [10]:
english_on_english = trigram_cross_entropy(english_test, english_model, english_lambda)
english_on_spanish = trigram_cross_entropy(spanish_test, english_model, english_lambda)
english_on_czech = trigram_cross_entropy(czech_test, english_model, english_lambda)
english_on_polish = trigram_cross_entropy(polish_test, english_model, english_lambda)

spanish_on_english = trigram_cross_entropy(english_test, spanish_model, spanish_lambda)
spanish_on_spanish = trigram_cross_entropy(spanish_test, spanish_model, spanish_lambda)
spanish_on_czech = trigram_cross_entropy(czech_test, spanish_model, spanish_lambda)
spanish_on_polish = trigram_cross_entropy(polish_test, spanish_model, spanish_lambda)

czech_on_english = trigram_cross_entropy(english_test, czech_model, czech_lambda)
czech_on_spanish = trigram_cross_entropy(spanish_test, czech_model, czech_lambda)
czech_on_czech = trigram_cross_entropy(czech_test, czech_model, czech_lambda)
czech_on_polish = trigram_cross_entropy(polish_test, czech_model, czech_lambda)

polish_on_english = trigram_cross_entropy(english_test, polish_model, polish_lambda)
polish_on_spanish = trigram_cross_entropy(spanish_test, polish_model, polish_lambda)
polish_on_czech = trigram_cross_entropy(czech_test, polish_model, polish_lambda)
polish_on_polish = trigram_cross_entropy(polish_test, polish_model, polish_lambda)

print(f"Cross-entropy of English model on English test set: {english_on_english}")
print(f"Cross-entropy of English model on Spanish test set: {english_on_spanish}")
print(f"Cross-entropy of English model on Czech test set: {english_on_czech}")
print(f"Cross-entropy of English model on Polish test set: {english_on_polish}")

print(f"Cross-entropy of Spanish model on English test set: {spanish_on_english}")
print(f"Cross-entropy of Spanish model on Spanish test set: {spanish_on_spanish}")
print(f"Cross-entropy of Spanish model on Czech test set: {spanish_on_czech}")
print(f"Cross-entropy of Spanish model on Polish test set: {spanish_on_polish}")

print(f"Cross-entropy of Czech model on English test set: {czech_on_english}")
print(f"Cross-entropy of Czech model on Spanish test set: {czech_on_spanish}")
print(f"Cross-entropy of Czech model on Czech test set: {czech_on_czech}")
print(f"Cross-entropy of Czech model on Polish test set: {czech_on_polish}")

print(f"Cross-entropy of Polish model on English test set: {polish_on_english}")
print(f"Cross-entropy of Polish model on Spanish test set: {polish_on_spanish}")
print(f"Cross-entropy of Polish model on Czech test set: {polish_on_czech}")
print(f"Cross-entropy of Polish model on Polish test set: {polish_on_polish}")

Cross-entropy of English model on English test set: 2.8252668680792663
Cross-entropy of English model on Spanish test set: 4.607452904125498
Cross-entropy of English model on Czech test set: 6.851012006758494
Cross-entropy of English model on Polish test set: 7.009073324362348
Cross-entropy of Spanish model on English test set: 4.482587143131362
Cross-entropy of Spanish model on Spanish test set: 3.748068161653823
Cross-entropy of Spanish model on Czech test set: 6.325674094222566
Cross-entropy of Spanish model on Polish test set: 6.9336714009449585
Cross-entropy of Czech model on English test set: 4.655686908675981
Cross-entropy of Czech model on Spanish test set: 5.254822215178093
Cross-entropy of Czech model on Czech test set: 3.2586809156602574
Cross-entropy of Czech model on Polish test set: 6.0341834358544455
Cross-entropy of Polish model on English test set: 4.4319282007933065
Cross-entropy of Polish model on Spanish test set: 4.877773029023663
Cross-entropy of Polish model on C

# 9. Write a function to identify language by comparing probabilities given by your models (the highest probability wins). This function should accept a string (of arbitrary length, containing more words or sentences) and return a list of pairs (probability, language) ordered by the probability, highest first.

In [16]:
# similar to trigram_cross_entropy function, but doesnt average the probability and makes the log probability as negative
def trigram_text_log_probability(text, model, _lambda):
    log_probability = 0.0

    for i in range(len(text) - 2):
        trigram = text[i:i + 3]
        probability = smoothed_trigram_probability(trigram, model, _lambda)
        log_probability += math.log2(probability)

    return log_probability

def identify_language(text):
    language_resources = {
        "English": (english_model, english_lambda, "en"),
        "Spanish": (spanish_model, spanish_lambda, "es"),
        "Czech": (czech_model, czech_lambda, "cs"),
        "Polish": (polish_model, polish_lambda, "pl")
    }

    # store the result pairs (probability, langauge)
    results = []

    # for all languages, calculate the given text's log probability of the text being in that language
    for language, (model, _lambda, language_code) in language_resources.items():
        tokenized_text = tokenize(text, language_code)
        log_probability = trigram_text_log_probability(tokenized_text, model, _lambda)
        probability = 2 ** (log_probability / (len(tokenized_text) - 2))
        results.append((probability, language))

    results.sort(reverse=True)
    return results

# Helper methods

In [20]:
# used for the OOV tokens
def oov_percentage(train_text, text):
    train_vocabulary = set(train_text.split())
    eval_tokens = text.split()
    oov_count = sum(token not in train_vocabulary for token in eval_tokens)
    return 100 * oov_count / len(eval_tokens)

print("Heldout OOV percentages:")
print(f"{oov_percentage(english_train, english_heldout)}, "
    f"{oov_percentage(spanish_train, spanish_heldout)}, "
    f"{oov_percentage(czech_train, czech_heldout)}, "
    f"{oov_percentage(polish_train, polish_heldout)}"
)

print("Test OOV percentages:")
print(
    f"{oov_percentage(english_train, english_test)}, "
    f"{oov_percentage(spanish_train, spanish_test)}, "
    f"{oov_percentage(czech_train, czech_test)}, "
    f"{oov_percentage(polish_train, polish_test)}"
)

Heldout OOV percentages:
2.0387590520079, 11.116467185552708, 11.86432417943907, 15.265243902439025
Test OOV percentages:
1.8741578119053253, 17.926047851390276, 13.9546783625731, 25.3140243902439


In [21]:
# used to print the Initial cross-entropy in the submission form
initial_lambda = 0.01

print(
    f"{trigram_cross_entropy(english_test, english_model, initial_lambda)}, "
    f"{trigram_cross_entropy(spanish_test, spanish_model, initial_lambda)}, "
    f"{trigram_cross_entropy(czech_test, czech_model, initial_lambda)}, "
    f"{trigram_cross_entropy(polish_test, polish_model, initial_lambda)}"
)

2.8252668680792663, 3.7897042233285463, 3.269991730717743, 3.3099891072696015


In [23]:
# this is the random sentence I'm using for the submission form
print(identify_language("This is a test sentence for the first assignment in the natural language processing course."))

[(0.1480427826642026, 'English'), (0.07323625746059036, 'Spanish'), (0.07081690459599795, 'Polish'), (0.059256375029275055, 'Czech')]
